In [3]:
!pip install kagglehub

  Using cached kagglehub-0.3.13-py3-none-any.whl.metadata (38 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached kagglehub-0.3.13-py3-none-any.whl (68 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")

print("Path to dataset files:", path)

/opt/anaconda3/envs/ecom-ws/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/olaf/.cache/kagglehub/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/versions/8


In [21]:
df_test = pd.read_csv(path+'/2019-Nov.csv')
df_test

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2
...,...,...,...,...,...,...,...,...,...
67501974,2019-11-30 23:59:58 UTC,view,15700137,2053013559733912211,NaN,NaN,277.74,532714000,02b4131c-0112-4231-aafa-ceaa08e77c1b
67501975,2019-11-30 23:59:58 UTC,view,28719425,2053013565639492569,apparel.shoes,baden,62.81,545223467,734c5eef-0742-4f8b-9d22-48f75b0bc359
67501976,2019-11-30 23:59:59 UTC,view,1004833,2053013555631882655,electronics.smartphone,samsung,167.03,557794415,6fecf566-ebb0-4e70-a243-cdc13ce044cb
67501977,2019-11-30 23:59:59 UTC,view,2701706,2053013563911439225,appliances.kitchen.refrigerators,samsung,566.27,531607492,368ddc8b-5db9-40fb-b7ff-b6582a1192c0


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, regexp_replace, to_timestamp, to_date, hour, date_format, when, lit
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pathlib import Path
import glob

# This function reads all csv files in the path
def read_csv_files(spark, pattern: str,schema):
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")
    files = [str(Path(f).resolve()) for f in files]
    return spark.read.option("header", True).schema(schema).csv(files)

# Create a Spark session to run spark sql
spark = (
    SparkSession.builder
    .appName("ecom-etl")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)
    .getOrCreate()
)

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("WARN")

raw_path = "data/raw/*.csv"
out_path = "data/processed/events_parquet"

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True),
])
df = read_csv_files(spark, raw_path ,schema )

def norm_str(c):
    x = lower(trim(col(c)))
    return when(x.isNull() | (x == "") | (x == "nan"), lit(None)).otherwise(x)

ALLOWED = ["view", "cart", "remove_from_cart", "purchase"]

clean = (
    df
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("brand", norm_str("brand"))
    .withColumn("category_code", norm_str("category_code"))
    .withColumn("user_session", trim(col("user_session")))
    .withColumn("event_time_str", regexp_replace(col("event_time"), r"\s+UTC$", ""))
    .withColumn("event_time", to_timestamp(col("event_time_str"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("event_date", to_date(col("event_time")))
    .withColumn("event_hour", hour(col("event_time")))
    .withColumn("event_month", date_format(col("event_time"), "yyyy-MM"))
    .filter(col("event_time").isNotNull())
    .filter(col("event_date").isNotNull())
    .filter(col("event_type").isin(ALLOWED))
    .filter(col("price").isNotNull() & (col("price") >= 0) & (col("price") < 1_000_000))
    .drop("event_time_str")
)

# Repartitioning to multiple parquets files 
clean = clean.repartition(120, "event_date")

(clean.write
 .mode("overwrite")
 .partitionBy("event_date")
 .parquet(out_path)
)

spark.stop()
print("Wrote parquet to:", out_path)

In [9]:
import glob
matches = glob.glob(raw_path)
if not matches:
    raise FileNotFoundError(f"No files matched raw_path={raw_path}. CWD={os.getcwd()}")
print("Matched files:", matches[:5], "...")


Matched files: ['data/raw/2019-Nov.csv', 'data/raw/2019-Oct.csv'] ...


In [15]:
events_path = "data/processed/events_parquet"

# 选一天（换成你实际有的日期，比如 2019-11-01）
one_day_path = f"{events_path}/event_date=2019-11-01"

df_day = spark.read.parquet(one_day_path)
df_day.show(20, truncate=False)
df_day.printSchema()
df_day.count()

+-------------------+----------+----------+-------------------+--------------------------------+--------+------+---------+------------------------------------+----------+-----------+
|event_time         |event_type|product_id|category_id        |category_code                   |brand   |price |user_id  |user_session                        |event_hour|event_month|
+-------------------+----------+----------+-------------------+--------------------------------+--------+------+---------+------------------------------------+----------+-----------+
|2019-10-31 20:00:00|view      |1003461   |2053013555631882655|electronics.smartphone          |xiaomi  |489.07|520088904|4d3b30da-a5e4-49df-b1a8-ba5943f1dd33|0         |2019-11    |
|2019-10-31 20:00:00|view      |5000088   |2053013566100866035|appliances.sewing_machine       |janome  |293.65|530496790|8e5f4f83-366c-4f70-860e-ca7417414283|0         |2019-11    |
|2019-10-31 20:00:01|view      |17302664  |2053013553853497655|NULL                  

1445360

In [23]:
ud = pd.read_parquet("data/processed/user_daily")
sf = pd.read_parquet("data/processed/session_funnel")

# 例：daily_high_events
ud["events_day"] = ud["views"] + ud["carts"] + ud["removes"] + ud["purchases"]
thr = ud["events_day"].quantile(0.999)
anoms = ud[ud["events_day"] >= thr].sort_values("events_day", ascending=False).head(200)

KeyboardInterrupt: 

In [5]:
import pandas as pd
df = pd.read_csv("/Users/olaf/bin/ecommerce_project/data/outputs/tables/anomalous_users.csv/part-00000-9a0ace6b-8f72-4711-9905-1516cd761358-c000.csv")
df[df['anomaly_type'] == 'funnel_mismatch']

,user_id,as_of_date,anomaly_type,score,evidence
600,512875426,2019-11-29,funnel_mismatch,709.0,"sum_carts_cnt=709, sum_purchases_cnt=0, mismat..."
601,513824664,2019-11-15,funnel_mismatch,299.0,"sum_carts_cnt=299, sum_purchases_cnt=0, mismat..."
602,521558076,2019-11-10,funnel_mismatch,292.0,"sum_carts_cnt=292, sum_purchases_cnt=0, mismat..."
603,515642515,2019-11-14,funnel_mismatch,237.0,"sum_carts_cnt=237, sum_purchases_cnt=0, mismat..."
604,513754375,2019-11-17,funnel_mismatch,203.0,"sum_carts_cnt=203, sum_purchases_cnt=0, mismat..."
...,...,...,...,...,...
778,516723213,2019-11-16,funnel_mismatch,50.0,"sum_carts_cnt=50, sum_purchases_cnt=0, mismatc..."
779,571337651,2019-11-16,funnel_mismatch,50.0,"sum_carts_cnt=50, sum_purchases_cnt=0, mismatc..."
780,517885000,2019-11-12,funnel_mismatch,50.0,"sum_carts_cnt=50, sum_purchases_cnt=0, mismatc..."
781,572070840,2019-11-21,funnel_mismatch,50.0,"sum_carts_cnt=50, sum_purchases_cnt=0, mismatc..."


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, regexp_replace, to_timestamp, to_date, hour, date_format, when, lit
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
from pathlib import Path
import glob

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("check-sizes")
    .getOrCreate()
)

print(spark.read.parquet("data/processed/user_daily").count())
print(spark.read.parquet("data/processed/session_funnel").count())


15095144
23017602
